In [ ]:

from google.colab import userdata
import os

token = userdata.get('KAGGLE_API_TOKEN')

os.environ['KAGGLE_API_TOKEN'] = token

print("Kaggle connected successfully!")


In [ ]:

!pip install -q --upgrade kaggle


In [ ]:

import os

dataset_path = "/content/plantvillage"

for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)

    print("\nFolder:", folder)
    print("Contents:", os.listdir(folder_path)[:10])


In [ ]:

import os
import random
import matplotlib.pyplot as plt
from PIL import Image

plt.figure(figsize=(12, 10))

categories = os.listdir(dataset_path)

for i in range(9):

    category = random.choice(categories)

    folder_path = os.path.join(dataset_path, category)

    image_name = random.choice(os.listdir(folder_path))

    image_path = os.path.join(folder_path, image_name)

    img = Image.open(image_path)

    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.title(category, fontsize=8)
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import tensorflow as tf

IMG_SIZE = (128, 128)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.30,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

temp_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.30,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)


In [ ]:

val_batches = tf.data.experimental.cardinality(temp_ds)

test_ds = temp_ds.take(val_batches // 2)

val_ds = temp_ds.skip(val_batches // 2)

print("Training batches:", tf.data.experimental.cardinality(train_ds).numpy())

print("Validation batches:", tf.data.experimental.cardinality(val_ds).numpy())

print("Testing batches:", tf.data.experimental.cardinality(test_ds).numpy())


In [ ]:

normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(
    lambda x, y: (normalization_layer(x), y)
)

val_ds = val_ds.map(
    lambda x, y: (normalization_layer(x), y)
)

test_ds = test_ds.map(
    lambda x, y: (normalization_layer(x), y)
)

print("Normalization completed!")


In [ ]:

from tensorflow.keras import layers, models


In [ ]:

model = models.Sequential([

    layers.Input(shape=(128, 128, 3)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),

    layers.Dropout(0.5),

    layers.Dense(15, activation='softmax')

])


In [ ]:

model.compile(

    optimizer='adam',

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)

print("CNN model compiled successfully!")


In [ ]:

model.summary()


In [ ]:

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)


In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Test Accuracy:", test_accuracy * 100, "%")
print("Test Loss:", test_loss)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

y_true = []
y_pred = []

for images, labels in test_ds:

    predictions = model.predict(images, verbose=0)

    predicted_classes = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_classes)

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(16, 14))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Plant Leaf Disease Detection - Confusion Matrix")

plt.xticks(rotation=90)
plt.yticks(rotation=0)

plt.show()

In [ ]:
# Recreate image array from the already loaded image
img_array = tf.keras.utils.img_to_array(img)

img_array = np.array(img_array, dtype=np.float32)
img_array = img_array / 255.0

img_array = np.expand_dims(img_array, axis=0)

print("Image shape:", img_array.shape)

# Predict again
prediction = model.predict(img_array, verbose=0)

print("\nTOP 5 PREDICTIONS\n")

top_5_indices = np.argsort(prediction[0])[-5:][::-1]

for index in top_5_indices:
    print("Plant:", plant_types[index])
    print("Disease:", class_names[index])
    print("Confidence:",
          round(float(prediction[0][index] * 100), 2), "%")
    print("----------------------")

In [ ]:
from collections import Counter

plant_counts = Counter(plant_types)

print("Images available for each plant:\n")

for plant, count in plant_counts.items():
    print(plant, ":", count)

In [ ]:
import os

plant_image_counts = {
    "Pepper": 0,
    "Potato": 0,
    "Tomato": 0
}

for folder in class_names:

    folder_path = os.path.join(dataset_path, folder)

    image_count = len([
        f for f in os.listdir(folder_path)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])

    if folder.startswith("Pepper"):
        plant_image_counts["Pepper"] += image_count

    elif folder.startswith("Potato"):
        plant_image_counts["Potato"] += image_count

    elif folder.startswith("Tomato"):
        plant_image_counts["Tomato"] += image_count

print("Images available for each plant:\n")

for plant, count in plant_image_counts.items():
    print(plant, ":", count)

In [ ]:
def convert_to_plant_type(images, labels):
    plant_labels = tf.gather(
        tf.constant([0, 0, 1, 1, 1,
                     2, 2, 2, 2, 2,
                     2, 2, 2, 2, 2]),
        labels
    )

    return images, plant_labels

train_plant_ds = train_ds.map(convert_to_plant_type)
val_plant_ds = val_ds.map(convert_to_plant_type)
test_plant_ds = test_ds.map(convert_to_plant_type)

plant_class_names = ["Pepper", "Potato", "Tomato"]

print("Plant categories:", plant_class_names)

In [ ]:
from tensorflow.keras import layers, models

plant_model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(3, activation='softmax')
])

plant_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

plant_model.summary()

In [ ]:
history_plant = plant_model.fit(
    train_plant_ds,
    validation_data=val_plant_ds,
    epochs=10
)

In [ ]:
plant_test_loss, plant_test_accuracy = plant_model.evaluate(
    test_plant_ds
)

print("Plant Classifier Test Accuracy:",
      round(plant_test_accuracy * 100, 2), "%")

In [ ]:
plant_model.save("plant_classifier.keras")

print("Plant classifier saved successfully!")

In [ ]:
from google.colab import files
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Upload image
uploaded = files.upload()

if uploaded:

    image_path = next(iter(uploaded))

    # Load image
    img = tf.keras.utils.load_img(
        image_path,
        target_size=(128, 128)
    )

    img_array = tf.keras.utils.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Step 1: Identify plant
    plant_prediction = plant_model.predict(
        img_array, verbose=0
    )

    plant_index = np.argmax(plant_prediction[0])
    plant_confidence = plant_prediction[0][plant_index]

    plant = plant_class_names[plant_index]

    plt.imshow(img)
    plt.axis("off")
    plt.show()

    print("Plant:", plant)
    print("Plant Confidence:",
          round(float(plant_confidence * 100), 2), "%")

    # Step 2: Check confidence
    if plant_confidence < 0.80:

        print("\nUnsupported or uncertain plant!")
        print("Cannot predict disease reliably.")

    else:

        # Step 3: Predict disease
        disease_prediction = model.predict(
            img_array, verbose=0
        )

        disease_index = np.argmax(disease_prediction[0])

        disease = class_names[disease_index]
        disease_plant = plant_types[disease_index]

        # Check plant consistency
        if disease_plant != plant:

            print("\nPrediction mismatch!")
            print("Please upload a clearer leaf image.")

        else:

            disease_confidence = (
                disease_prediction[0][disease_index] * 100
            )

            print("\nPredicted Disease:", disease)
            print("Disease Confidence:",
                  round(float(disease_confidence), 2), "%")

In [ ]:
print("PLANT CLASSIFIER RESULTS\n")

for i, plant in enumerate(plant_class_names):
    print(
        plant, ":",
        round(float(plant_prediction[0][i] * 100), 2),
        "%"
    )

In [ ]:
import os
import random

folder = os.path.join(
    dataset_path,
    "Tomato_Septoria_leaf_spot"
)

image_name = random.choice(os.listdir(folder))
image_path = os.path.join(folder, image_name)

img = tf.keras.utils.load_img(
    image_path,
    target_size=(128, 128)
)

img_array = tf.keras.utils.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

plant_prediction = plant_model.predict(img_array, verbose=0)

for i, plant in enumerate(plant_class_names):
    print(plant, ":", round(
        float(plant_prediction[0][i] * 100), 2
    ), "%")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

y_true = []
y_pred = []

for images, labels in test_plant_ds:
    predictions = plant_model.predict(images, verbose=0)

    predicted_labels = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_labels)

print(classification_report(
    y_true,
    y_pred,
    target_names=plant_class_names
))

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

folder = os.path.join(
    dataset_path,
    "Potato___Early_blight"
)

image_name = random.choice(os.listdir(folder))
image_path = os.path.join(folder, image_name)

img = tf.keras.utils.load_img(
    image_path,
    target_size=(128, 128)
)

img_array = tf.keras.utils.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict plant
plant_prediction = plant_model.predict(img_array, verbose=0)
plant_index = np.argmax(plant_prediction[0])
plant = plant_class_names[plant_index]

# Predict disease
disease_prediction = model.predict(img_array, verbose=0)
disease_index = np.argmax(disease_prediction[0])
disease = class_names[disease_index]

plt.imshow(img)
plt.axis("off")
plt.show()

print("Actual Plant: Potato")
print("Actual Disease: Early Blight")
print("Predicted Plant:", plant)
print("Predicted Disease:", disease)

In [ ]:

import os
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Select Tomato Septoria image
folder = os.path.join(
    dataset_path,
    "Tomato_Septoria_leaf_spot"
)

image_name = random.choice(os.listdir(folder))
image_path = os.path.join(folder, image_name)

# Load and preprocess image
img = tf.keras.utils.load_img(
    image_path,
    target_size=(128, 128)
)

img_array = tf.keras.utils.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Step 1: Predict plant
plant_prediction = plant_model.predict(
    img_array,
    verbose=0
)

plant_index = np.argmax(plant_prediction[0])
plant = plant_class_names[plant_index]

# Step 2: Predict disease
disease_prediction = model.predict(
    img_array,
    verbose=0
)

disease_index = np.argmax(disease_prediction[0])
disease = class_names[disease_index]

# Display image
plt.imshow(img)
plt.axis("off")
plt.show()

# Display results
print("Actual Plant: Tomato")
print("Actual Disease: Septoria Leaf Spot")

print("\nPredicted Plant:", plant)
print("Predicted Disease:", disease)

print("\nPlant Confidence:",
      round(float(plant_prediction[0][plant_index] * 100), 2), "%")

print("Disease Confidence:",
      round(float(disease_prediction[0][disease_index] * 100), 2), "%")


In [ ]:

import os
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Select Pepper image
folder = os.path.join(
    dataset_path,
    "Pepper__bell___Bacterial_spot"
)

image_name = random.choice(os.listdir(folder))
image_path = os.path.join(folder, image_name)

# Load image
img = tf.keras.utils.load_img(
    image_path,
    target_size=(128, 128)
)

img_array = tf.keras.utils.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict plant
plant_prediction = plant_model.predict(img_array, verbose=0)
plant_index = np.argmax(plant_prediction[0])
plant = plant_class_names[plant_index]

# Predict disease
disease_prediction = model.predict(img_array, verbose=0)
disease_index = np.argmax(disease_prediction[0])
disease = class_names[disease_index]

# Display image
plt.imshow(img)
plt.axis("off")
plt.show()

# Results
print("Actual Plant: Pepper")
print("Actual Disease: Bacterial Spot")

print("\nPredicted Plant:", plant)
print("Predicted Disease:", disease)

print("\nPlant Confidence:",
      round(float(plant_prediction[0][plant_index] * 100), 2), "%")

print("Disease Confidence:",
      round(float(disease_prediction[0][disease_index] * 100), 2), "%")


In [ ]:

import os
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Select a healthy Tomato leaf
folder = os.path.join(
    dataset_path,
    "Tomato_healthy"
)

image_name = random.choice(os.listdir(folder))
image_path = os.path.join(folder, image_name)

# Load image
img = tf.keras.utils.load_img(
    image_path,
    target_size=(128, 128)
)

img_array = tf.keras.utils.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Plant prediction
plant_prediction = plant_model.predict(img_array, verbose=0)
plant_index = np.argmax(plant_prediction[0])
plant = plant_class_names[plant_index]

# Disease prediction
disease_prediction = model.predict(img_array, verbose=0)
disease_index = np.argmax(disease_prediction[0])
disease = class_names[disease_index]

# Display image
plt.imshow(img)
plt.axis("off")
plt.show()

# Results
print("Actual Plant: Tomato")
print("Actual Disease: Healthy")

print("\nPredicted Plant:", plant)
print("Predicted Disease:", disease)

print("\nPlant Confidence:",
      round(float(plant_prediction[0][plant_index] * 100), 2), "%")

print("Disease Confidence:",
      round(float(disease_prediction[0][disease_index] * 100), 2), "%")


In [ ]:

import os
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Select healthy Potato leaf
folder = os.path.join(
    dataset_path,
    "Potato___healthy"
)

image_name = random.choice(os.listdir(folder))
image_path = os.path.join(folder, image_name)

# Load image
img = tf.keras.utils.load_img(
    image_path,
    target_size=(128, 128)
)

img_array = tf.keras.utils.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict plant
plant_prediction = plant_model.predict(img_array, verbose=0)
plant_index = np.argmax(plant_prediction[0])
plant = plant_class_names[plant_index]

# Predict disease
disease_prediction = model.predict(img_array, verbose=0)
disease_index = np.argmax(disease_prediction[0])
disease = class_names[disease_index]

# Display image
plt.imshow(img)
plt.axis("off")
plt.show()

# Results
print("Actual Plant: Potato")
print("Actual Disease: Healthy")

print("\nPredicted Plant:", plant)
print("Predicted Disease:", disease)

print("\nPlant Confidence:",
      round(float(plant_prediction[0][plant_index] * 100), 2), "%")

print("Disease Confidence:",
      round(float(disease_prediction[0][disease_index] * 100), 2), "%")


In [ ]:

import os
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Select healthy Pepper leaf
folder = os.path.join(
    dataset_path,
    "Pepper__bell___healthy"
)

image_name = random.choice(os.listdir(folder))
image_path = os.path.join(folder, image_name)

# Load image
img = tf.keras.utils.load_img(
    image_path,
    target_size=(128, 128)
)

img_array = tf.keras.utils.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict plant
plant_prediction = plant_model.predict(img_array, verbose=0)
plant_index = np.argmax(plant_prediction[0])
plant = plant_class_names[plant_index]

# Predict disease
disease_prediction = model.predict(img_array, verbose=0)
disease_index = np.argmax(disease_prediction[0])
disease = class_names[disease_index]

# Display image
plt.imshow(img)
plt.axis("off")
plt.show()

# Results
print("Actual Plant: Pepper")
print("Actual Disease: Healthy")

print("\nPredicted Plant:", plant)
print("Predicted Disease:", disease)

print("\nPlant Confidence:",
      round(float(plant_prediction[0][plant_index] * 100), 2), "%")

print("Disease Confidence:",
      round(float(disease_prediction[0][disease_index] * 100), 2), "%")


In [ ]:
# Save plant classifier
plant_model.save("plant_classifier.keras")

# Save disease classifier
model.save("plant_disease_model.keras")

print("Both models saved successfully!")

In [ ]:
from google.colab import files

files.download("plant_classifier.keras")
files.download("plant_disease_model.keras")

In [ ]:
from google.colab import files
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Upload image
uploaded = files.upload()

if uploaded:
    image_path = next(iter(uploaded))

    # Load image
    img = tf.keras.utils.load_img(
        image_path,
        target_size=(128, 128)
    )

    img_array = tf.keras.utils.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Predict plant
    plant_prediction = plant_model.predict(img_array, verbose=0)
    plant_index = np.argmax(plant_prediction[0])
    plant_confidence = float(plant_prediction[0][plant_index])
    plant = plant_class_names[plant_index]

    # Display image
    plt.imshow(img)
    plt.axis("off")
    plt.show()

    print("Predicted Plant:", plant)
    print("Plant Confidence:",
          round(plant_confidence * 100, 2), "%")

    # Check confidence
    if plant_confidence < 0.80:
        print("\nPlant prediction is uncertain!")
        print("Cannot predict disease reliably.")

    else:
        # Predict disease
        disease_prediction = model.predict(img_array, verbose=0)
        disease_index = np.argmax(disease_prediction[0])
        disease = class_names[disease_index]
        disease_plant = plant_types[disease_index]

        if disease_plant != plant:
            print("\nPrediction mismatch!")
            print("Please upload a clearer leaf image.")

        else:
            disease_confidence = float(
                disease_prediction[0][disease_index]
            )

            print("\nPredicted Disease:", disease)
            print("Disease Confidence:",
                  round(disease_confidence * 100, 2), "%")

In [ ]:
from google.colab import files
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Upload Rose leaf image
uploaded = files.upload()

if uploaded:
    image_path = next(iter(uploaded))

    # Load image
    img = tf.keras.utils.load_img(
        image_path,
        target_size=(128, 128)
    )

    img_array = tf.keras.utils.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Plant prediction
    plant_prediction = plant_model.predict(img_array, verbose=0)

    plant_index = np.argmax(plant_prediction[0])
    plant = plant_class_names[plant_index]
    plant_confidence = float(plant_prediction[0][plant_index])

    # Display image
    plt.imshow(img)
    plt.axis("off")
    plt.show()

    print("Predicted Plant:", plant)
    print("Plant Confidence:",
          round(plant_confidence * 100, 2), "%")

    # Existing confidence check
    if plant_confidence < 0.80:
        print("\nUnsupported or uncertain plant!")
        print("Cannot predict disease reliably.")

    else:
        disease_prediction = model.predict(img_array, verbose=0)

        disease_index = np.argmax(disease_prediction[0])
        disease = class_names[disease_index]
        disease_confidence = float(
            disease_prediction[0][disease_index]
        )

        print("\nPredicted Disease:", disease)
        print("Disease Confidence:",
              round(disease_confidence * 100, 2), "%")

In [ ]:
top_5 = np.argsort(prediction[0])[::-1][:5]

for i in top_5:
    print(
        class_names[i],
        ":", round(prediction[0][i] * 100, 2), "%"
    )

In [ ]:
import os

septoria_path = os.path.join(
    dataset_path,
    "Tomato_Septoria_leaf_spot"
)

print("Septoria images:", len(os.listdir(septoria_path)))

In [ ]:
import os
import random
import tensorflow as tf
import matplotlib.pyplot as plt

septoria_path = os.path.join(
    dataset_path,
    "Tomato_Septoria_leaf_spot"
)

image_name = random.choice(os.listdir(septoria_path))
image_path = os.path.join(septoria_path, image_name)

img = tf.keras.utils.load_img(image_path)

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.title("Tomato Septoria Leaf Spot")
plt.axis("off")
plt.show()

In [ ]:
img = tf.keras.utils.load_img(
    image_path,
    target_size=(128, 128)
)

img_array = tf.keras.utils.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array, verbose=0)

index = np.argmax(prediction[0])

print("Actual disease: Tomato Septoria Leaf Spot")
print("Predicted disease:", class_names[index])
print("Confidence:", round(prediction[0][index] * 100, 2), "%")

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf

septoria_path = os.path.join(
    dataset_path,
    "Tomato_Septoria_leaf_spot"
)

# Select 20 random Septoria images
image_names = random.sample(
    os.listdir(septoria_path), 20
)

images = []

for name in image_names:
    path = os.path.join(septoria_path, name)

    img = tf.keras.utils.load_img(
        path,
        target_size=(128, 128)
    )

    img_array = tf.keras.utils.img_to_array(img) / 255.0
    images.append(img_array)

# Predict all 20 images
images = np.array(images)
predictions = model.predict(images, verbose=0)

# Display results
correct = 0

for i, prediction in enumerate(predictions):
    predicted_index = np.argmax(prediction)
    predicted_name = class_names[predicted_index]

    if predicted_name == "Tomato_Septoria_leaf_spot":
        correct += 1

    print(i + 1, predicted_name,
          round(prediction[predicted_index] * 100, 2), "%")

print("\nCorrect predictions:", correct, "/ 20")